# Five-fold out-of-fold performance analysis

This notebook analyzes within-HPO generalization using patient-level out-of-fold (OOF) predictions. Each eligible patient is predicted once by a model that was not trained on that patient. HPO-specific metrics from pooled OOF predictions are treated as the primary results, and a dedicated sensitivity section compares them with the unweighted mean of the five fold-specific AUROCs.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")
sns.set_theme(style="whitegrid", context="notebook")

## Load revised OOF results

Run the updated `finetune.py` before this notebook. The earlier `5fold_cv` files contain fold-level metrics and must not be substituted because patient-level pooled AUROC cannot be reconstructed from them.

In [ ]:
working_dir = Path.cwd()
result_dir = (working_dir if working_dir.name == "3.retrain_regression"
              else working_dir / "3.retrain_regression")
performance_path = result_dir / "finetune_per_hpo_all_lookback_years_5fold_oof.csv"
predictions_path = result_dir / "patient_level_out_of_fold_predictions.csv"
mean_fold_path = result_dir / "finetune_per_hpo_all_lookback_years_5fold_mean_auc.csv"

missing_files = [
    str(path) for path in [performance_path, predictions_path, mean_fold_path]
    if not path.exists()
]
if missing_files:
    raise FileNotFoundError(
        "Run the updated finetune.py first. Missing revised OOF output(s):\n"
        + "\n".join(missing_files)
    )

performance = pd.read_csv(performance_path)
predictions = pd.read_csv(predictions_path)
mean_fold_performance = pd.read_csv(mean_fold_path)
print(f"HPO-level rows: {len(performance):,}")
print(f"Patient-level OOF rows: {len(predictions):,}")
print(f"Mean-fold HPO-level rows: {len(mean_fold_performance):,}")

## Data and cross-validation checks

In [ ]:
performance

In [ ]:
required_performance_columns = {
    "src_hpo_id", "hpo", "patient_count", "hpo_positive_cases",
    "lookback_year", "aou_roc_auc", "weighted_roc_auc", "ori_roc_auc",
    "aou_pr_auc", "weighted_pr_auc", "ori_pr_auc",
    "aou_brier_score", "weighted_brier_score", "ori_brier_score",
    "aou_ece", "weighted_ece", "ori_ece",
}
missing = required_performance_columns - set(performance.columns)
assert not missing, f"Missing performance columns: {sorted(missing)}"
assert "cv_fold" not in performance.columns, (
    "This appears to be a fold-level file. Use the revised 5fold_oof output."
)
assert performance["hpo"].notna().all(), "Missing HPO identifiers."
assert performance["lookback_year"].notna().all(), "Missing lookback years."
assert not performance.duplicated(["hpo", "lookback_year"]).any(), (
    "Expected one row per HPO and lookback year."
)

performance["negative_cases"] = (performance["patient_count"] -
                                   performance["hpo_positive_cases"])
performance["minority_class_count"] = performance[[
    "hpo_positive_cases", "negative_cases"
]].min(axis=1)
performance["cv_eligible"] = performance["aou_roc_auc"].notna()

coverage = (performance.groupby("lookback_year")
            .agg(total_hpos=("hpo", "nunique"),
                 eligible_hpos=("cv_eligible", "sum"),
                 total_patients=("patient_count", "sum"))
            .reset_index())
coverage["excluded_hpos"] = coverage["total_hpos"] - coverage["eligible_hpos"]
display(coverage)

excluded_hpos = performance.loc[~performance["cv_eligible"], [
    "lookback_year", "hpo", "patient_count", "hpo_positive_cases",
    "negative_cases", "minority_class_count",
]].sort_values(["lookback_year", "hpo"])
display(excluded_hpos)

In [ ]:
required_prediction_columns = {
    "oof_row_id", "src_hpo_id", "visit_label", "lookback_year", "cv_fold",
    "aou_probability", "weighted_probability", "original_probability",
}
missing = required_prediction_columns - set(predictions.columns)
assert not missing, f"Missing patient-level columns: {sorted(missing)}"
assert not predictions.duplicated(["lookback_year", "oof_row_id"]).any(), (
    "At least one observation received multiple OOF predictions."
)
probability_columns = ["aou_probability", "weighted_probability", "original_probability"]
assert predictions[probability_columns].notna().all().all(), "Missing probabilities."
assert predictions[probability_columns].apply(lambda x: x.between(0, 1).all()).all(), (
    "At least one probability falls outside [0, 1]."
)
assert predictions["cv_fold"].between(0, 4).all(), "Unexpected fold identifier."
fold_coverage = predictions.groupby(["lookback_year", "src_hpo_id"])["cv_fold"].nunique()
assert fold_coverage.eq(5).all(), "An eligible HPO does not contain all five held-out folds."

expected = (performance.loc[performance["cv_eligible"]]
            .groupby("lookback_year")["patient_count"].sum().sort_index())
observed = predictions.groupby("lookback_year").size().sort_index()
prediction_coverage = pd.concat([expected.rename("expected"),
                                 observed.rename("observed")], axis=1)
prediction_coverage["difference"] = (prediction_coverage["observed"] -
                                       prediction_coverage["expected"])
display(prediction_coverage)
assert prediction_coverage["difference"].eq(0).all(), "OOF prediction coverage mismatch."

## HPO-level performance summary

The primary summaries are medians and IQRs across eligible HPOs, giving each site equal influence. Means are included only as supplementary descriptive statistics.

In [ ]:
eligible = performance[performance["cv_eligible"]].copy()
model_labels = {"aou": "AoU unweighted",
                "weighted": "AoU HPO-weighted",
                "ori": "Original"}
metric_labels = {"roc_auc": "AUROC", "pr_auc": "PR-AUC",
                 "brier_score": "Brier score", "ece": "ECE"}
summary_rows = []
for lookback_year, year_df in eligible.groupby("lookback_year"):
    for model, model_label in model_labels.items():
        for metric, metric_label in metric_labels.items():
            values = year_df[f"{model}_{metric}"].dropna()
            summary_rows.append({
                "lookback_year": int(lookback_year), "model": model_label,
                "metric": metric_label, "hpo_count": len(values),
                "mean": values.mean(), "sd": values.std(ddof=1),
                "median": values.median(),
                "q1": values.quantile(0.25), "q3": values.quantile(0.75),
                "minimum": values.min(), "maximum": values.max(),
            })
summary = pd.DataFrame(summary_rows)
summary["mean_sd"] = summary.apply(
    lambda row: f"{row['mean']:.3f} ({row['sd']:.3f})", axis=1
)
summary["median_iqr"] = summary.apply(
    lambda row: f"{row['median']:.3f} ({row['q1']:.3f}–{row['q3']:.3f})", axis=1
)
manuscript_table = (
    summary.pivot(
        index=["lookback_year", "metric"],
        columns="model",
        values=["mean_sd"],
    )
    .swaplevel(0, 1, axis=1)
    .sort_index(axis=1)
    .reset_index()
)
display(manuscript_table[manuscript_table['lookback_year'] == 5])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
for ax, metric in zip(axes.flat, metric_labels.values()):
    plot_df = summary[summary["metric"] == metric]
    for model, model_df in plot_df.groupby("model", sort=False):
        model_df = model_df.sort_values("lookback_year")
        x = model_df["lookback_year"].to_numpy()
        median = model_df["median"].to_numpy()
        q1 = model_df["q1"].to_numpy()
        q3 = model_df["q3"].to_numpy()
        ax.plot(x, median, marker="o", label=model)
        ax.fill_between(x, q1, q3, alpha=0.15)
    ax.set(title=f"HPO-level {metric}", xlabel="Lookback years", ylabel=metric)
    ax.set_xticks(sorted(eligible["lookback_year"].unique()))
axes[0, 0].legend(frameon=True)
plt.show()
plt.savefig("performance_metrics_5years.png", dpi=300, bbox_inches="tight")

## Pooled OOF versus unweighted mean-fold AUROC

The pooled method concatenates all held-out predictions for an HPO before calculating AUROC. The mean-fold method calculates AUROC separately in each held-out fold and then takes the unweighted arithmetic mean of the five fold AUROCs. Positive differences below mean that the mean-fold estimate is higher than the pooled estimate.

In [ ]:
required_mean_fold_columns = {
    'lookback_year', 'src_hpo_id',
    'aou_roc_auc_mean', 'weighted_roc_auc_mean',
    'original_roc_auc_mean',
}
missing = required_mean_fold_columns - set(mean_fold_performance.columns)
assert not missing, f'Missing mean-fold columns: {sorted(missing)}'
assert not mean_fold_performance.duplicated(
    ['lookback_year', 'src_hpo_id']
).any(), 'Expected one mean-fold row per HPO and lookback year.'

method_comparison_parts = []
method_model_columns = {
    'AoU unweighted': ('aou_roc_auc', 'aou_roc_auc_mean'),
    'AoU HPO-weighted': ('weighted_roc_auc', 'weighted_roc_auc_mean'),
    'Original': ('ori_roc_auc', 'original_roc_auc_mean'),
}

for model_label, (pooled_column, mean_fold_column) in method_model_columns.items():
    model_comparison = (
        eligible[[
            'lookback_year', 'hpo', 'patient_count', pooled_column
        ]]
        .merge(
            mean_fold_performance[[
                'lookback_year', 'src_hpo_id', mean_fold_column
            ]],
            left_on=['lookback_year', 'hpo'],
            right_on=['lookback_year', 'src_hpo_id'],
            how='inner',
            validate='one_to_one',
        )
        .rename(columns={
            pooled_column: 'pooled_oof_auc',
            mean_fold_column: 'mean_fold_auc',
        })
    )
    model_comparison['model'] = model_label
    model_comparison['mean_fold_minus_pooled'] = (
        model_comparison['mean_fold_auc']
        - model_comparison['pooled_oof_auc']
    )
    method_comparison_parts.append(model_comparison)

method_comparison = pd.concat(method_comparison_parts, ignore_index=True)

method_comparison_summary = (
    method_comparison.groupby(['lookback_year', 'model'], as_index=False)
    .agg(
        hpo_count=('hpo', 'nunique'),
        pooled_mean=('pooled_oof_auc', 'mean'),
        pooled_median=('pooled_oof_auc', 'median'),
        mean_fold_mean=('mean_fold_auc', 'mean'),
        mean_fold_median=('mean_fold_auc', 'median'),
        mean_difference=('mean_fold_minus_pooled', 'mean'),
        median_difference=('mean_fold_minus_pooled', 'median'),
        minimum_difference=('mean_fold_minus_pooled', 'min'),
        maximum_difference=('mean_fold_minus_pooled', 'max'),
    )
)
display(method_comparison_summary)

In [ ]:
method_comparison_output_dir = result_dir / 'analysis_outputs'
method_comparison_output_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
for axis, (model_label, model_df) in zip(
    axes, method_comparison.groupby('model', sort=False)
):
    sns.scatterplot(
        data=model_df,
        x='pooled_oof_auc',
        y='mean_fold_auc',
        hue='lookback_year',
        palette='viridis',
        alpha=0.7,
        ax=axis,
    )
    lower = min(model_df['pooled_oof_auc'].min(), model_df['mean_fold_auc'].min())
    upper = max(model_df['pooled_oof_auc'].max(), model_df['mean_fold_auc'].max())
    axis.plot([lower, upper], [lower, upper], '--', color='firebrick')
    axis.set(
        title=model_label,
        xlabel='Pooled patient-level OOF AUROC',
        ylabel='Unweighted mean-fold AUROC',
        xlim=(lower, upper),
        ylim=(lower, upper),
    )
    axis.set_aspect('equal', adjustable='box')
    if axis is not axes[-1]:
        axis.get_legend().remove()
fig.savefig(
    method_comparison_output_dir / 'pooled_vs_mean_fold_auc_agreement.png',
    dpi=300, bbox_inches='tight', facecolor='white',
)
plt.show()

method_summary_long = method_comparison.melt(
    id_vars=['lookback_year', 'hpo', 'model'],
    value_vars=['pooled_oof_auc', 'mean_fold_auc'],
    var_name='method',
    value_name='auc',
)
method_summary_long['method'] = method_summary_long['method'].map({
    'pooled_oof_auc': 'Pooled OOF',
    'mean_fold_auc': 'Mean of five folds',
})
method_trend = (
    method_summary_long.groupby(
        ['lookback_year', 'model', 'method'], as_index=False
    )['auc'].mean()
)
comparison_grid = sns.relplot(
    data=method_trend,
    x='lookback_year',
    y='auc',
    hue='method',
    col='model',
    kind='line',
    marker='o',
    facet_kws={'sharey': True},
    height=4.5,
    aspect=1.1,
)
comparison_grid.set_axis_labels('Lookback years', 'Mean HPO-level AUROC')
comparison_grid.set_titles('{col_name}')
comparison_grid.figure.savefig(
    method_comparison_output_dir / 'pooled_vs_mean_fold_auc_by_lookback.png',
    dpi=300, bbox_inches='tight', facecolor='white',
)
plt.show()

# The methods overlap on the AUROC scale, so show their paired difference
# in basis points (1 basis point = 0.0001 AUROC).
difference_plot_data = method_comparison.copy()
difference_plot_data['difference_basis_points'] = (
    10_000 * difference_plot_data['mean_fold_minus_pooled']
)
fig, ax = plt.subplots(figsize=(11, 6), constrained_layout=True)
sns.boxplot(
    data=difference_plot_data,
    x='lookback_year',
    y='difference_basis_points',
    hue='model',
    showfliers=False,
    ax=ax,
)
sns.stripplot(
    data=difference_plot_data,
    x='lookback_year',
    y='difference_basis_points',
    hue='model',
    dodge=True,
    alpha=0.25,
    size=2.5,
    legend=False,
    ax=ax,
)
ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.set(
    xlabel='Lookback years',
    ylabel='Mean-fold minus pooled OOF AUROC (basis points)',
    title='Paired difference between AUROC aggregation methods',
)
ax.legend(title='Model', frameon=True)
fig.savefig(
    method_comparison_output_dir / 'mean_fold_minus_pooled_auc_difference.png',
    dpi=300, bbox_inches='tight', facecolor='white',
)
plt.show()

display(
    method_comparison_summary[[
        'lookback_year', 'model', 'mean_difference',
        'median_difference', 'minimum_difference', 'maximum_difference',
    ]].assign(
        mean_difference_basis_points=lambda frame: 10_000 * frame['mean_difference'],
        median_difference_basis_points=lambda frame: 10_000 * frame['median_difference'],
    )
)

## Five-year model comparisons

Each point represents one HPO; point size represents the full HPO sample size. Higher AUROC and PR-AUC are better, whereas lower Brier score and ECE are better.

In [ ]:
primary_lookback = 5
primary = eligible[eligible["lookback_year"] == primary_lookback].copy()

def comparison_plot(ax, data, x, y, x_label, y_label, title):
    plot_data = data.dropna(subset=[x, y])
    lower = min(plot_data[x].min(), plot_data[y].min())
    upper = max(plot_data[x].max(), plot_data[y].max())
    padding = max((upper - lower) * 0.05, 0.01)
    limits = (lower - padding, upper + padding)
    sns.scatterplot(data=plot_data, x=x, y=y, size="patient_count",
                    sizes=(25, 400), alpha=0.65, legend=False, ax=ax)
    ax.plot(limits, limits, color="firebrick", linestyle="--")
    ax.set(xlim=limits, ylim=limits, xlabel=x_label, ylabel=y_label, title=title)
    ax.set_aspect("equal", adjustable="box")

five_year_metrics = {
    "AUROC": "roc_auc",
    "PR-AUC": "pr_auc",
    "Brier score": "brier_score",
    "ECE": "ece",
}
pairwise_models = [
    ("ori", "aou", "Original", "AoU unweighted"),
    ("ori", "weighted", "Original", "AoU HPO-weighted"),
    ("aou", "weighted", "AoU unweighted", "AoU HPO-weighted"),
]
plot_output_dir = (result_dir / "analysis_outputs" /
                   "five_year_model_comparison_plots")
plot_output_dir.mkdir(parents=True, exist_ok=True)

for metric_label, metric_suffix in five_year_metrics.items():
    direction = ("higher is better" if metric_suffix in {"roc_auc", "pr_auc"}
                 else "lower is better")
    for x_model, y_model, x_name, y_name in pairwise_models:
        fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
        comparison_plot(
            ax,
            primary,
            f"{x_model}_{metric_suffix}",
            f"{y_model}_{metric_suffix}",
            f"{x_name} {metric_label}",
            f"{y_name} {metric_label}",
            f"Five-year {metric_label}: {y_name} vs {x_name}",
        )
        plot_path = plot_output_dir / (
            f"five_year_{metric_suffix}_{y_model}_vs_{x_model}.png"
        )
        fig.savefig(plot_path, dpi=600, bbox_inches="tight", facecolor="white")
        plt.show()

## One-year model comparisons

Each point represents one HPO; point size represents the full HPO sample size. Higher AUROC and PR-AUC are better, whereas lower Brier score and ECE are better.

In [ ]:
primary_lookback = 1
primary = eligible[eligible["lookback_year"] == primary_lookback].copy()

def comparison_plot(ax, data, x, y, x_label, y_label, title):
    plot_data = data.dropna(subset=[x, y])
    lower = min(plot_data[x].min(), plot_data[y].min())
    upper = max(plot_data[x].max(), plot_data[y].max())
    padding = max((upper - lower) * 0.05, 0.01)
    limits = (lower - padding, upper + padding)
    sns.scatterplot(data=plot_data, x=x, y=y, size="patient_count",
                    sizes=(25, 400), alpha=0.65, legend=False, ax=ax)
    ax.plot(limits, limits, color="firebrick", linestyle="--")
    ax.set(xlim=limits, ylim=limits, xlabel=x_label, ylabel=y_label, title=title)
    ax.set_aspect("equal", adjustable="box")

one_year_metrics = {
    "AUROC": "roc_auc",
    "PR-AUC": "pr_auc",
    "Brier score": "brier_score",
    "ECE": "ece",
}
pairwise_models = [
    ("ori", "aou", "Original", "AoU unweighted"),
    ("ori", "weighted", "Original", "AoU HPO-weighted"),
    ("aou", "weighted", "AoU unweighted", "AoU HPO-weighted"),
]
plot_output_dir = (result_dir / "analysis_outputs" /
                   "one_year_model_comparison_plots")
plot_output_dir.mkdir(parents=True, exist_ok=True)

for metric_label, metric_suffix in one_year_metrics.items():
    direction = ("higher is better" if metric_suffix in {"roc_auc", "pr_auc"}
                 else "lower is better")
    for x_model, y_model, x_name, y_name in pairwise_models:
        fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
        comparison_plot(
            ax,
            primary,
            f"{x_model}_{metric_suffix}",
            f"{y_model}_{metric_suffix}",
            f"{x_name} {metric_label}",
            f"{y_name} {metric_label}",
            f"One-year {metric_label}: {y_name} vs {x_name}",
        )
        plot_path = plot_output_dir / (
            f"one_year_{metric_suffix}_{y_model}_vs_{x_model}.png"
        )
        fig.savefig(plot_path, dpi=600, bbox_inches="tight", facecolor="white")
        plt.show()

## Paired HPO differences and sparse-outcome sensitivity

In [ ]:
difference_rows = []
for year, year_df in eligible.groupby("lookback_year"):
    comparisons = {
        "AoU unweighted minus original": year_df["aou_roc_auc"] - year_df["ori_roc_auc"],
        "AoU HPO-weighted minus original": year_df["weighted_roc_auc"] - year_df["ori_roc_auc"],
        "AoU HPO-weighted minus unweighted": year_df["weighted_roc_auc"] - year_df["aou_roc_auc"],
    }
    for comparison, differences in comparisons.items():
        difference_rows.append({
            "lookback_year": int(year), "comparison": comparison,
            "hpo_count": differences.notna().sum(),
            "median_auc_difference": differences.median(),
            "hpos_higher": differences.gt(0).sum(),
            "hpos_lower": differences.lt(0).sum(),
            "hpos_equal": differences.eq(0).sum(),
        })
paired_summary = pd.DataFrame(difference_rows)
display(paired_summary)

sensitivity_rows = []
for year, year_df in eligible.groupby("lookback_year"):
    for label, subset in [("Five-fold eligible", year_df),
                          ("At least 10 per class", year_df[year_df["minority_class_count"].ge(10)])]:
        sensitivity_rows.append({
            "lookback_year": int(year), "analysis": label,
            "hpo_count": subset["hpo"].nunique(),
            "aou_auc_median": subset["aou_roc_auc"].median(),
            "weighted_auc_median": subset["weighted_roc_auc"].median(),
            "original_auc_median": subset["ori_roc_auc"].median(),
        })
sensitivity_summary = pd.DataFrame(sensitivity_rows)
display(sensitivity_summary)

## DeLong results

The pooled OOF point estimates are appropriate cross-validated metrics. DeLong tests on OOF predictions are secondary because they do not incorporate uncertainty from overlapping training sets; site-wise significance counts are descriptive and unadjusted for multiple comparisons.

In [ ]:
comparison_specs = [
    ("AoU unweighted vs original", "delong_p_value_ori_aou", "aou_roc_auc", "ori_roc_auc"),
    ("AoU HPO-weighted vs original", "delong_p_value_ori_weighted", "weighted_roc_auc", "ori_roc_auc"),
    ("AoU HPO-weighted vs unweighted", "delong_p_value_aou_weighted", "weighted_roc_auc", "aou_roc_auc"),
]
delong_rows = []
for year, year_df in eligible.groupby("lookback_year"):
    for label, p_column, first_auc, second_auc in comparison_specs:
        estimable = year_df[p_column].notna()
        significant = estimable & year_df[p_column].lt(0.05)
        difference = year_df[first_auc] - year_df[second_auc]
        delong_rows.append({
            "lookback_year": int(year), "comparison": label,
            "estimable_hpos": int(estimable.sum()),
            "p_below_0.05": int(significant.sum()),
            "significantly_higher_auc": int((significant & difference.gt(0)).sum()),
            "significantly_lower_auc": int((significant & difference.lt(0)).sum()),
        })
delong_summary = pd.DataFrame(delong_rows)
display(delong_summary)

## Export tables and draft Results text

In [ ]:
output_dir = result_dir / "analysis_outputs"
output_dir.mkdir(exist_ok=True)
summary.to_csv(output_dir / "fivefold_oof_hpo_summary.csv", index=False)
manuscript_table.to_csv(output_dir / "fivefold_oof_manuscript_table.csv", index=False)
coverage.to_csv(output_dir / "fivefold_oof_hpo_coverage.csv", index=False)
excluded_hpos.to_csv(output_dir / "fivefold_oof_excluded_hpos.csv", index=False)
paired_summary.to_csv(output_dir / "fivefold_oof_paired_auc_summary.csv", index=False)
sensitivity_summary.to_csv(output_dir / "fivefold_oof_sensitivity.csv", index=False)
delong_summary.to_csv(output_dir / "fivefold_oof_delong_descriptive_summary.csv", index=False)
method_comparison.to_csv(
    output_dir / "pooled_vs_mean_fold_auc_by_hpo.csv", index=False
)
method_comparison_summary.to_csv(
    output_dir / "pooled_vs_mean_fold_auc_summary.csv", index=False
)

first_year = summary[(summary["lookback_year"] == 1) & (summary["metric"] == "AUROC")].set_index("model")
last_year = summary[(summary["lookback_year"] == 7) & (summary["metric"] == "AUROC")].set_index("model")
first_coverage = coverage.loc[coverage["lookback_year"].eq(1)].iloc[0]
results_text = (
    f"Five-fold within-HPO cross-validation included {int(first_coverage['eligible_hpos'])} "
    f"of {int(first_coverage['total_hpos'])} HPOs. At the 1-year lookback, median "
    f"HPO-specific OOF AUROC was {first_year.loc['AoU unweighted', 'median_iqr']} "
    f"for the unweighted AoU model, {first_year.loc['AoU HPO-weighted', 'median_iqr']} "
    f"for the HPO-weighted model, and {first_year.loc['Original', 'median_iqr']} "
    f"for the original model. At the 7-year lookback, corresponding median AUROCs "
    f"were {last_year.loc['AoU unweighted', 'median_iqr']}, "
    f"{last_year.loc['AoU HPO-weighted', 'median_iqr']}, and "
    f"{last_year.loc['Original', 'median_iqr']}, respectively."
)
print(results_text)
print(f"\nTables saved to: {output_dir}")